<a href="https://colab.research.google.com/github/TalCordova/RMBA_SemB26_TC_SC/blob/main/Final_Project_SC_TC_fit_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Training Notebook

**Research Methods for Business Analytics** | Semester B, 2026 | Coller School of Management, Tel Aviv University

**Authors:**
- Saar Cohen, 213499312
- Tal Cordova, 203868187

---

## Overview

This notebook trains and evaluates classification models to predict whether a customer will purchase a product (`BUYER_FLAG = 1`), so the business can decide whom to target with a marketing contact.

It applies the feature engineering decisions and modelling strategy derived from the [EDA Notebook](Course_Final_Project_TC_SC.ipynb).

**Pipeline:**
1. **Feature engineering** — recency-weighted aggregation of yearly fare/points; drop uninformative flags
2. **Custom profit metric** — reflects the asymmetric payoff: TP = +$78.4, FP = −$15.2
3. **Baseline model comparison** — Logistic Regression, Decision Tree, Random Forest, XGBoost evaluated with 5-fold stratified CV

In [1]:
import pandas as pd
from sklearn.metrics import make_scorer

In [2]:
# --- Mount Google Drive - run only in Google Colab ---
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [3]:
# -- Run this cell only if you are using Google Colab ---
tal_path_ffp_train = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/ffp_train.csv'
tal_path_reviews_train = '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/reviews_training.csv'

saar_path_ffp_train ='/content/drive/MyDrive/Research Methods for Business Analytics/ffp_train.csv'
saar_pth_ffp_reviews = '/content/drive/MyDrive/Research Methods for Business Analytics/reviews_training.csv'
# --- Load training data for EDA ---
ffp_train = pd.read_csv(tal_path_ffp_train)
reviews_train = pd.read_csv(tal_path_reviews_train)

# --- Initial shape check ---
print(f'ffp_train shape: {ffp_train.shape}')
print(f'reviews_train shape: {reviews_train.shape}')

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/PhD - TAU/Research Methods for Business Analytics/ffp_train.csv'

In [4]:
## load local
# --- Load training data for EDA ---
ffp_train = pd.read_csv('ffp_train.csv')
reviews_train = pd.read_csv('reviews_training.csv')

# --- Initial shape check ---
print(f'ffp_train shape: {ffp_train.shape}')
print(f'reviews_train shape: {reviews_train.shape}')


ffp_train shape: (20000, 23)
reviews_train shape: (1499, 2001)


In [5]:
ffp_rollout = pd.read_csv('ffp_rollout_X.csv')
reviews_rollout = pd.read_csv('reviews_rollout.csv')

print(f'ffp_rollout shape: {ffp_rollout.shape}')
print(f'reviews_rollout shape: {reviews_rollout.shape}')

ffp_rollout shape: (20000, 23)
reviews_rollout shape: (1501, 2001)


## 1. Feature Engineering

We apply two transformations informed by the EDA:

1. **Recency-weighted aggregation of `FARE` and `POINTS`** — the five yearly columns are highly collinear (r > 0.92). Rather than drop them or use a simple mean, we compute a weighted sum that gives more weight to recent years (Y1 = 1.0 down to Y5 = 0.2). This preserves magnitude while reducing the feature space from 10 columns to 2.

2. **Drop `ANIMAL_FLAG`** — the EDA showed it has virtually no discriminative power (near-zero lift and negligible correlation with the target).

In [6]:
# --- Feature Engineering: Recency-Weighted Fare & Points ---
RECENCY_WEIGHTS = [1.0, 0.8, 0.6, 0.4, 0.2]  # Y1 (most recent) → Y5 (oldest)

fare_cols   = [f'FARE_L_Y{i}'   for i in range(1, 6)]
points_cols = [f'POINTS_L_Y{i}' for i in range(1, 6)]

ffp_train['FARE_WEIGHTED']   = sum(
    w * ffp_train[c] for w, c in zip(RECENCY_WEIGHTS, fare_cols)
)
ffp_train['POINTS_WEIGHTED'] = sum(
    w * ffp_train[c] for w, c in zip(RECENCY_WEIGHTS, points_cols)
)

# --- Impute negative user grade with the median of all other user grades ---
median_grade = ffp_train.loc[ffp_train['CUSTOMER_GRADE'] >= 0, 'CUSTOMER_GRADE'].median()
ffp_train.loc[ffp_train['CUSTOMER_GRADE'] < 0, 'CUSTOMER_GRADE'] = median_grade

# --- Drop original yearly columns and ANIMAL_FLAG ---
cols_to_drop = fare_cols + points_cols + ['ANIMAL_FLAG']
ffp_train.drop(columns=cols_to_drop, inplace=True)

# --- Also drop EDA-only aggregates created earlier (FARE_MEAN, POINTS_MEAN) ---
ffp_train.drop(columns=['FARE_MEAN', 'POINTS_MEAN'], inplace=True, errors='ignore')

# --- Verify final feature set ---
print('Remaining columns:', ffp_train.columns.tolist())
print('Shape:', ffp_train.shape)

Remaining columns: ['ID', 'CUSTOMER_GRADE', 'STATUS_PANTINUM', 'STATUS_GOLD', 'STATUS_SILVER', 'NUM_DEAL', 'LAST_DEAL', 'ADVANCE_PURCHASE', 'ATT_FLAG', 'CREDIT_FLAG', 'RELATED_FLAG', 'BUYER_FLAG', 'FARE_WEIGHTED', 'POINTS_WEIGHTED']
Shape: (20000, 14)


## 2. Custom Profit Metric

The dataset is heavily imbalanced (~9.5:1 negative-to-positive ratio), and the business payoff is asymmetric:

| Outcome | Description | Value |
|---------|-------------|-------|
| True Positive (TP) | Contact a buyer → sale | **+$78.4** |
| False Positive (FP) | Contact a non-buyer → wasted outreach | **−$15.2** |
| True Negative / False Negative | No contact | $0 |

Standard accuracy or AUC do not capture this asymmetry. We therefore define a **per-customer average profit** score and optimise for it directly.

The **classification threshold** (0.1624) was derived from the TP/FP payoff ratio: a contact is profitable if `P(buyer) ≥ 15.2 / (15.2 + 78.4) ≈ 0.162`.

In [7]:
THRESHOLD = 0.1624

feature_cols = [c for c in ffp_train.columns if c not in ['ID', 'BUYER_FLAG']]

X = ffp_train[feature_cols]
y = ffp_train['BUYER_FLAG']

# --- Custom profit scorer ---
def avg_profit(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    TP = ((y_pred == 1) & (y_true == 1)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    return (TP * 78.4 - FP * 15.2) / len(y_true)

profit_scorer = make_scorer(avg_profit, response_method='predict_proba')

## 3. Baseline Model Comparison

We evaluate four model families with 5-fold stratified cross-validation. Stratified folds preserve the 9.5% minority class proportion in every split.

Each model is wrapped in a `Pipeline`. Logistic Regression includes a `StandardScaler`; tree-based models do not need one.

Metrics reported per fold:
- **Avg Profit** - the primary metric (per-customer profit at threshold 0.1624)
- **AUC** - threshold-independent discrimination measure
- **Accuracy** - included for reference, but misleading given class imbalance

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.metrics import make_scorer, roc_auc_score, accuracy_score
import numpy as np

# --- Constants ---
THRESHOLD = 0.1624
RANDOM_STATE = 42

feature_cols = [c for c in ffp_train.columns if c not in ['ID', 'BUYER_FLAG']]
X = ffp_train[feature_cols]
y = ffp_train['BUYER_FLAG']

# --- Custom profit scorer ---
def avg_profit_scorer(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(int)
    TP = ((y_pred == 1) & (y_true == 1)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    return (TP * 78.4 - FP * 15.2) / len(y_true)

profit_scorer = make_scorer(avg_profit_scorer, response_method='predict_proba')
auc_scorer    = make_scorer(roc_auc_score, response_method='predict_proba')

scoring = {
    'avg_profit': profit_scorer,
    'auc':        auc_scorer,
    'accuracy':   'accuracy',
}

# --- CV strategy: stratified to preserve 9.5% minority class in each fold ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# --- Model definitions ---
# LR needs scaling; trees do not — each wrapped in its own Pipeline
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Decision Tree': Pipeline([
        ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
        # n_jobs=-1 omitted — causes deadlocks in Colab
    ]),
    'XGBoost': Pipeline([
        ('clf', XGBClassifier(
            n_estimators=100,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            verbosity=0
        ))
    ]),
}

# --- Run CV for all models ---
results = {}

for name, pipeline in models.items():
    print(f'Running CV: {name}...')
    cv_results = cross_validate(
        pipeline, X, y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )
    results[name] = {
        'avg_profit_mean': cv_results['test_avg_profit'].mean(),
        'avg_profit_std':  cv_results['test_avg_profit'].std(),
        'auc_mean':        cv_results['test_auc'].mean(),
        'auc_std':         cv_results['test_auc'].std(),
        'accuracy_mean':   cv_results['test_accuracy'].mean(),
        'accuracy_std':    cv_results['test_accuracy'].std(),
    }
    print(f"  Avg Profit: {results[name]['avg_profit_mean']:.4f} ± {results[name]['avg_profit_std']:.4f}")
    print(f"  AUC:        {results[name]['auc_mean']:.4f} ± {results[name]['auc_std']:.4f}")
    print(f"  Accuracy:   {results[name]['accuracy_mean']:.4f} ± {results[name]['accuracy_std']:.4f}")
    print()

# --- Summary table ---
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('avg_profit_mean', ascending=False)
print('=== Baseline Model Comparison (sorted by Avg Profit) ===')
print(results_df.round(4))

Running CV: Logistic Regression...
  Avg Profit: 0.4170 ± 0.1150
  AUC:        0.6526 ± 0.0192
  Accuracy:   0.9050 ± 0.0000

Running CV: Decision Tree...
  Avg Profit: 0.0886 ± 0.0879
  AUC:        0.5487 ± 0.0066
  Accuracy:   0.8325 ± 0.0023

Running CV: Random Forest...
  Avg Profit: 0.9290 ± 0.2826
  AUC:        0.6471 ± 0.0205
  Accuracy:   0.9036 ± 0.0019

Running CV: XGBoost...
  Avg Profit: 1.0005 ± 0.1601
  AUC:        0.6504 ± 0.0287
  Accuracy:   0.9003 ± 0.0026

=== Baseline Model Comparison (sorted by Avg Profit) ===
                     avg_profit_mean  avg_profit_std  auc_mean  auc_std  \
XGBoost                       1.0005          0.1601    0.6504   0.0287   
Random Forest                 0.9290          0.2826    0.6471   0.0205   
Logistic Regression           0.4170          0.1150    0.6526   0.0192   
Decision Tree                 0.0886          0.0879    0.5487   0.0066   

                     accuracy_mean  accuracy_std  
XGBoost                     0.9003  

## 4. Hyperparaeter Tuning

`XGBoost` is the best model on all metrics we tested (custom avergae user profit, AUC and accuracy) - so we will use it as our model.

In this section we will tune it, using nested cross-validation.

### 4.1 Why nested cross-validation?

The baseline comparison above reused the same 5 `StratifiedKFold` folds to score every model with **fixed** hyperparameters. There was no tuning involved, so no risk of optimism bias. If we now search over XGBoost's hyperparameters using those same 5 folds and report the best fold's score, the result would leak information: the folds used to *select* hyperparameters would be the same folds used to *report* performance, and the search would tend to find whatever configuration happens to fit those particular folds best (optimism bias / data leakage).

Nested cross-validation avoids this. An **outer** `StratifiedKFold` provides test folds that are never touched during the hyperparameter search. For each outer fold, an **inner** `StratifiedKFold` (run only on that fold's training data) drives a `RandomizedSearchCV`. The outer-fold scores then give an unbiased estimate of how the *tuning procedure*, and not a single fixed model, performs on unseen data.

### 4.2 Why optimize on `avg_profit`, not AUC?

AUC measures global ranking quality and is threshold-independent, but the business decision is **not**, we always classify at the fixed `THRESHOLD = 0.1624`. Hyperparameters such as `scale_pos_weight` shift the calibration of the predicted probabilities; a configuration that improves AUC can still push more probabilities above 0.1624 and hurt profit (more false positives), or vice versa. Since `avg_profit` is what the business actually cares about at that specific cutoff, the inner search refits on `avg_profit`; AUC and accuracy are still computed on every outer test fold and reported alongside it as diagnostics.

In [16]:
# --- Nested CV: RandomizedSearchCV (inner) wrapped in cross_validate (outer) ---
from sklearn.model_selection import RandomizedSearchCV

xgb_pipeline = Pipeline([
    ('clf', XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0))
])

xgb_param_dist = {
    'clf__n_estimators':     [100, 200, 300],
    'clf__max_depth':        [3, 4, 5, 6],
    'clf__learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'clf__subsample':        [0.7, 0.8, 0.9, 1.0],
    'clf__colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'clf__min_child_weight': [1, 3, 5, 7],
    'clf__gamma':            [0, 0.1, 0.5],
    'clf__scale_pos_weight': [1, 5, 9.5, 15],  # 9.5 ~= actual 18100/1900 imbalance ratio
}

# Separate from cell 11's `cv` (identical spec) so that cell stays independently re-runnable
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring=scoring,
    refit='avg_profit',
    cv=inner_cv,
    random_state=RANDOM_STATE,
    n_jobs=1,  # n_jobs=-1 causes deadlocks in Colab (see Random Forest note above)
    verbose=0,
)

# cross_validate clones + fits xgb_search on each outer-train fold (running the full
# inner search inside it), then scores the refit best_estimator_ on the outer-test fold.
nested_cv_results = cross_validate(
    xgb_search, X, y,
    cv=outer_cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=1,
)

print('Best hyperparameters per outer fold:')
for fold_idx, fitted_search in enumerate(nested_cv_results['estimator']):
    print(f'  Fold {fold_idx}: {fitted_search.best_params_}')

results['XGBoost (Tuned, Nested CV)'] = {
    'avg_profit_mean': nested_cv_results['test_avg_profit'].mean(),
    'avg_profit_std':  nested_cv_results['test_avg_profit'].std(),
    'auc_mean':        nested_cv_results['test_auc'].mean(),
    'auc_std':         nested_cv_results['test_auc'].std(),
    'accuracy_mean':   nested_cv_results['test_accuracy'].mean(),
    'accuracy_std':    nested_cv_results['test_accuracy'].std(),
}

results_df = pd.DataFrame(results).T.sort_values('avg_profit_mean', ascending=False)
print('\n=== Updated Model Comparison (sorted by Avg Profit) ===')
print(results_df.round(4))

Best hyperparameters per outer fold:
  Fold 0: {'clf__subsample': 0.8, 'clf__scale_pos_weight': 1, 'clf__n_estimators': 100, 'clf__min_child_weight': 1, 'clf__max_depth': 3, 'clf__learning_rate': 0.05, 'clf__gamma': 0, 'clf__colsample_bytree': 0.7}
  Fold 1: {'clf__subsample': 0.8, 'clf__scale_pos_weight': 1, 'clf__n_estimators': 100, 'clf__min_child_weight': 1, 'clf__max_depth': 3, 'clf__learning_rate': 0.05, 'clf__gamma': 0, 'clf__colsample_bytree': 0.7}
  Fold 2: {'clf__subsample': 0.8, 'clf__scale_pos_weight': 1, 'clf__n_estimators': 100, 'clf__min_child_weight': 1, 'clf__max_depth': 3, 'clf__learning_rate': 0.05, 'clf__gamma': 0, 'clf__colsample_bytree': 0.7}
  Fold 3: {'clf__subsample': 0.8, 'clf__scale_pos_weight': 1, 'clf__n_estimators': 100, 'clf__min_child_weight': 1, 'clf__max_depth': 3, 'clf__learning_rate': 0.05, 'clf__gamma': 0, 'clf__colsample_bytree': 0.7}
  Fold 4: {'clf__subsample': 0.8, 'clf__scale_pos_weight': 1, 'clf__n_estimators': 100, 'clf__min_child_weight': 1,

### 4.3 Results interpretation

Running this section locally (sklearn 1.6.1, xgboost 3.3.0) produced:

| | avg_profit | AUC | accuracy |
|---|---|---|---|
| XGBoost (Tuned, Nested CV) | 1.59 ± 0.25 | 0.671 ± 0.023 | 0.905 ± 0.001 |
| XGBoost (baseline, untuned) | 1.00 ± 0.16 | 0.650 ± 0.029 | 0.900 ± 0.003 |

Tuning lifted average profit by roughly **+59%** over the untuned baseline (1.00 → 1.59), with AUC also improving (0.650 → 0.671). Both metrics moved in the same direction, so the profit gain isn't just an artifact of probability recalibration crossing the fixed 0.1624 threshold.


> 💡 Exact numbers will shift slightly when re-run on Colab, since XGBoost's tree   construction isn't bit-identical across library versions even with a fixed `random_state`.

All 5 outer folds' inner searches converged on the **same** hyperparameters: shallower trees (`max_depth=3` vs. the untuned default of 6), a lower `learning_rate` (0.05 vs. default 0.3), row/column subsampling (`subsample=0.8`, `colsample_bytree=0.7`), and — notably, `scale_pos_weight=1`, i.e. **no** extra reweighting for the 9.5:1 class imbalance. That the search consistently avoided reweighting suggests XGBoost's own gradient boosting already handles this imbalance reasonably well at the chosen threshold, and that explicit reweighting mostly just shifts the operating point without a net profit benefit here. All 5 folds agreeing (rather than scattering across the search space) is itself a good indication that the selection isn't noise-driven.

In [10]:
# --- Final hyperparameter search on full data (the model to carry forward) ---
final_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring=scoring,
    refit='avg_profit',
    cv=outer_cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=0,
)
final_search.fit(X, y)

print('Final selected hyperparameters (XGBoost):')
for k, v in final_search.best_params_.items():
    print(f'  {k}: {v}')
print(f"Inner CV avg_profit at these params: {final_search.best_score_:.4f}")

xgb_final_model = final_search.best_estimator_

Final selected hyperparameters (XGBoost):
  clf__subsample: 0.8
  clf__scale_pos_weight: 1
  clf__n_estimators: 100
  clf__min_child_weight: 1
  clf__max_depth: 3
  clf__learning_rate: 0.05
  clf__gamma: 0
  clf__colsample_bytree: 0.7
Inner CV avg_profit at these params: 1.6003


In [ ]:
# --- Custom asymmetric-loss XGBoost: bake the $78.4/$15.2 cost ratio into the gradient ---
from scipy.special import expit

FN_OVER_FP_RATIO = 78.4 / 15.2  # ~5.158 -- how much harder a missed buyer is penalized than a wasted contact

def asymmetric_profit_objective(y_true, y_pred):
    # y_pred is the raw margin (pre-sigmoid); XGBoost custom objectives operate in margin space.
    p = expit(y_pred)
    grad = (1 - y_true) * p - FN_OVER_FP_RATIO * y_true * (1 - p)
    hess = ((1 - y_true) + FN_OVER_FP_RATIO * y_true) * p * (1 - p)
    return grad, hess

# Freeze the nested-CV-selected hyperparameters from xgb_final_model, swap in only the objective
frozen_params = xgb_final_model.named_steps['clf'].get_params()
frozen_params['objective'] = asymmetric_profit_objective

custom_loss_pipeline = Pipeline([
    ('clf', XGBClassifier(**frozen_params))
])

# Standard (non-nested) 5-fold CV — no hyperparameter search here, just re-scoring a fixed config
custom_loss_cv_results = cross_validate(
    custom_loss_pipeline, X, y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

results['XGBoost (Custom Asymmetric Loss)'] = {
    'avg_profit_mean': custom_loss_cv_results['test_avg_profit'].mean(),
    'avg_profit_std':  custom_loss_cv_results['test_avg_profit'].std(),
    'auc_mean':        custom_loss_cv_results['test_auc'].mean(),
    'auc_std':         custom_loss_cv_results['test_auc'].std(),
    'accuracy_mean':   custom_loss_cv_results['test_accuracy'].mean(),
    'accuracy_std':    custom_loss_cv_results['test_accuracy'].std(),
}

results_df = pd.DataFrame(results).T.sort_values('avg_profit_mean', ascending=False)
print('=== Final Model Comparison (sorted by Avg Profit) ===')
print(results_df.round(4))

### 4.4 Where should the cost asymmetry live: the loss, or the threshold?

So far the asymmetry has lived in the threshold: we train a plain model, then apply the 0.1624 cutoff derived from the $78.4/$15.2 payoffs. The alternative is to bake the asymmetry into training itself, by weighting false negatives ~5.158x harder in the loss function, and see what happens if we then apply that same 0.1624 cutoff.

| | avg_profit | AUC | accuracy |
|---|---|---|---|
| XGBoost (Custom Asymmetric Loss) | **−6.25 ± 0.02** | **0.673 ± 0.023** | 0.848 ± 0.008 |
| XGBoost (Tuned, Nested CV) | 1.59 ± 0.25 | 0.671 ± 0.023 | 0.905 ± 0.001 |

Profit collapses, but AUC is the best of any model here. That split tells us what happened: the model still ranks buyers above non-buyers correctly (AUC), but training it to fear missing a buyer pushed almost every predicted probability up, so 99.8% of customers now clear the old 0.1624 cutoff. 

The asymmetry moved into the loss, but we were still judging it with a cutoff that assumes the asymmetry *isn't* there yet — so we double-counted it, and the model contacts almost everyone.

This also explains why the earlier hyperparameter search always picked `scale_pos_weight = 1`: this hyperparameter is weighting the positibe class, but in a model that the asymmetry lives in the threshold, there is no reason to reweight.

So - now we ask, what is the right threshold for the asymmetric loss? And will it score better than the standard loss model? 

In [12]:
# --- Threshold optimization: can a recalibrated cutoff rescue the custom-loss model's profit? ---
from sklearn.model_selection import cross_val_predict

oof_proba = cross_val_predict(custom_loss_pipeline, X, y, cv=cv, method='predict_proba', n_jobs=1)[:, 1]

thresholds = np.round(np.linspace(0.10, 0.95, 86), 2)  # 0.10 to 0.95 in steps of 0.01
profits = np.array([avg_profit_scorer(y, oof_proba, threshold=t) for t in thresholds])

best_idx = int(np.argmax(profits))
best_threshold = float(thresholds[best_idx])
best_profit = float(profits[best_idx])

print(f'Best threshold: {best_threshold:.2f}')
print(f'Max avg_profit (pooled out-of-fold) at this threshold: {best_profit:.4f}')

# Recompute fold-level profit/accuracy at the optimized threshold for a results-table-consistent
# mean/std (AUC is threshold-independent, so it is carried over unchanged)
fold_profits, fold_accuracies = [], []
for _, test_idx in cv.split(X, y):
    y_te = y.iloc[test_idx].values
    p_te = oof_proba[test_idx]
    fold_profits.append(avg_profit_scorer(y_te, p_te, threshold=best_threshold))
    fold_accuracies.append(accuracy_score(y_te, (p_te >= best_threshold).astype(int)))

results['XGBoost (Custom Asymmetric Loss)'] = {
    'avg_profit_mean': np.mean(fold_profits),
    'avg_profit_std':  np.std(fold_profits),
    'auc_mean':        results['XGBoost (Custom Asymmetric Loss)']['auc_mean'],
    'auc_std':         results['XGBoost (Custom Asymmetric Loss)']['auc_std'],
    'accuracy_mean':   np.mean(fold_accuracies),
    'accuracy_std':    np.std(fold_accuracies),
}

results_df = pd.DataFrame(results).T.sort_values('avg_profit_mean', ascending=False)
print('\n=== Final Comparison (Custom Asymmetric Loss now using its optimized threshold) ===')
print(results_df.round(4))

Best threshold: 0.51
Max avg_profit (pooled out-of-fold) at this threshold: 1.6066

=== Final Comparison (Custom Asymmetric Loss now using its optimized threshold) ===
                                  avg_profit_mean  avg_profit_std  auc_mean  \
XGBoost (Custom Asymmetric Loss)           1.6066          0.2471    0.6748   
XGBoost (Tuned, Nested CV)                 1.6003          0.2652    0.6742   
XGBoost                                    1.0005          0.1601    0.6504   
Random Forest                              0.9290          0.2826    0.6471   
Logistic Regression                        0.4170          0.1150    0.6526   
Decision Tree                              0.0886          0.0879    0.5487   

                                  auc_std  accuracy_mean  accuracy_std  
XGBoost (Custom Asymmetric Loss)   0.0200         0.8502        0.0080  
XGBoost (Tuned, Nested CV)         0.0217         0.9046        0.0004  
XGBoost                            0.0287         0.9003   

We can see that the threshold now is 0.51 - this makes sense. When we move the asymmetry to the loss, it no longer needs to be accounted for in the threshold, so it just goes back to being balanced.

#### 4.4.1 Sanity check: is 0.1624 actually the right threshold for `xgb_final_model`?

Paradigm 2 just got its threshold swept and re-fit. Paradigm 1 never has — it's been using the formula-derived 0.1624 the whole notebook, untested. Before comparing the two paradigms, we check the same way: sweep the threshold on `xgb_final_model`'s own out-of-fold predictions and see if 0.1624 is actually where its profit peaks, or if it's leaving money on the table.

In [13]:
# --- Sanity check: is 0.1624 actually the best threshold for the standard-loss model too? ---
oof_proba_standard = cross_val_predict(xgb_final_model, X, y, cv=cv, method='predict_proba', n_jobs=1)[:, 1]

profits_standard = np.array([avg_profit_scorer(y, oof_proba_standard, threshold=t) for t in thresholds])

best_idx_standard = int(np.argmax(profits_standard))
best_threshold_standard = float(thresholds[best_idx_standard])
best_profit_standard = float(profits_standard[best_idx_standard])

print(f'Best threshold (standard loss): {best_threshold_standard:.2f}')
print(f'Max avg_profit (pooled out-of-fold) at this threshold: {best_profit_standard:.4f}')
print(f"Profit at the fixed business threshold (0.1624): {avg_profit_scorer(y, oof_proba_standard, threshold=THRESHOLD):.4f}")

# Same fold-level recompute as the custom-loss check above, for a results-table-consistent mean/std
fold_profits_standard, fold_accuracies_standard = [], []
for _, test_idx in cv.split(X, y):
    y_te = y.iloc[test_idx].values
    p_te = oof_proba_standard[test_idx]
    fold_profits_standard.append(avg_profit_scorer(y_te, p_te, threshold=best_threshold_standard))
    fold_accuracies_standard.append(accuracy_score(y_te, (p_te >= best_threshold_standard).astype(int)))

results['XGBoost (Standard Loss, Empirical Threshold)'] = {
    'avg_profit_mean': np.mean(fold_profits_standard),
    'avg_profit_std':  np.std(fold_profits_standard),
    'auc_mean':        results['XGBoost (Tuned, Nested CV)']['auc_mean'],
    'auc_std':         results['XGBoost (Tuned, Nested CV)']['auc_std'],
    'accuracy_mean':   np.mean(fold_accuracies_standard),
    'accuracy_std':    np.std(fold_accuracies_standard),
}

results_df = pd.DataFrame(results).T.sort_values('avg_profit_mean', ascending=False)
print('\n=== Final Comparison (standard-loss model now also shown at its own empirical threshold) ===')
print(results_df.round(4))

Best threshold (standard loss): 0.15
Max avg_profit (pooled out-of-fold) at this threshold: 1.5959
Profit at the fixed business threshold (0.1624): 1.6003

=== Final Comparison (standard-loss model now also shown at its own empirical threshold) ===
                                              avg_profit_mean  avg_profit_std  \
XGBoost (Custom Asymmetric Loss)                       1.6066          0.2471   
XGBoost (Tuned, Nested CV)                             1.6003          0.2652   
XGBoost (Standard Loss, Empirical Threshold)           1.5959          0.2575   
XGBoost                                                1.0005          0.1601   
Random Forest                                          0.9290          0.2826   
Logistic Regression                                    0.4170          0.1150   
Decision Tree                                          0.0886          0.0879   

                                              auc_mean  auc_std  \
XGBoost (Custom Asymmetric Loss)   

**Result:** We can see that the best threshold is 1.5, which gives a profit of 0.15959 - so our hypothesis holds. The 0.1642 is a good estimate, and it was not a guess, it was based on the business problem definition. That is also how the hyperparameters were chosen.

The quetion remains is - where should the asymmetry live?

#### 4.4.2 Conclusion: where should the asymmetry live?

Once both models get a threshold that actually matches them, they land in the same place:

| Approach | avg_profit | AUC | accuracy |
|---|---|---|---|
| Asymmetry in the threshold (0.1624), standard loss — `xgb_final_model` | 1.6003 ± 0.2652 | 0.6742 ± 0.0217 | 0.9046 ± 0.0004 |
| Asymmetry in the loss, threshold refit to 0.51 | 1.6066 ± 0.2471 | 0.6748 ± 0.0200 | 0.8502 ± 0.0080 |

Same profit, within noise. We also checked the obvious follow-up question: was 0.1624 actually right for the standard-loss model, or just assumed? It holds up: sweeping its threshold finds nothing better (0.15 gives 1.5959, slightly worse than 0.1624's 1.6003).

So the asymmetry can live in either place and the business outcome doesn't change. What does change is how hard each version is to build and maintain:
- **Threshold version**: one line, `15.2/(15.2+78.4)`, using a standard loss function. If the payoffs change, recompute the line — no retraining.
- **Loss version**: a hand-written gradient/Hessian, frozen hyperparameters copied from the other model, and a threshold that has to be re-swept by hand. If the payoffs change, both the loss and the threshold need redoing.

Since they perform the same, we put the asymmetry where it's simplest: the threshold. **`xgb_final_model`** — standard loss, threshold 0.1624 — is the model we deploy.

## 5 Add Text Features

In this part, we will add the cell features from the [text mining notebook](C:\Users\talcordova\Projects\RMBA_SemB26_TC_SC\Course_Project_Text_Mining_TC_SC.ipynb) - we wil ladd two features:

1. `has_review` - a flag to indicate if a user has a review.
2. `positive_proba` - the probabiloty from the LR model that the user will have a positive review.

In [18]:
# --- Add Text Features: has_review & positive_proba ---

review_preds = pd.read_csv('reviews_train_preds.csv')  # columns: ID, rating

# Map ID -> predicted positive probability
proba_by_id = review_preds.set_index('ID')['rating']

# has_review: 1 if the customer has a review prediction, else 0
ffp_train['has_review'] = ffp_train['ID'].isin(proba_by_id.index).astype(int)

# positive_proba: predicted probability for reviewers; -1 sentinel for the rest.

ffp_train['positive_proba'] = ffp_train['ID'].map(proba_by_id).fillna(-1)

# --- Sanity checks ---
n_reviewers = ffp_train['has_review'].sum()
print(f'Customers with a review: {n_reviewers} / {len(ffp_train)}')
print(f'positive_proba range (reviewers): '
      f"[{ffp_train.loc[ffp_train['has_review'] == 1, 'positive_proba'].min():.4f}, "
      f"{ffp_train.loc[ffp_train['has_review'] == 1, 'positive_proba'].max():.4f}]")
print(f"Non-reviewer positive_proba unique values: "
      f"{ffp_train.loc[ffp_train['has_review'] == 0, 'positive_proba'].unique()}")
ffp_train[['ID', 'has_review', 'positive_proba']].head()

Customers with a review: 1499 / 20000
positive_proba range (reviewers): [0.0000, 1.0000]
Non-reviewer positive_proba unique values: [-1.]


,ID,has_review,positive_proba
0,1,0,-1.000000
1,2,0,-1.000000
2,3,1,0.201083
3,4,0,-1.000000
4,5,0,-1.000000


In [15]:
# --- Add Text Features to rollout ---

review_preds_rollout = pd.read_csv('reviews_rollout_preds.csv')  # columns: ID, rating

# Map ID -> predicted positive probability
proba_by_id = review_preds_rollout.set_index('ID')['rating']

# has_review: 1 if the customer has a review prediction, else 0
ffp_rollout['has_review'] = ffp_rollout['ID'].isin(proba_by_id.index).astype(int)

# positive_proba: predicted probability for reviewers; -1 sentinel for the rest.

ffp_rollout['positive_proba'] = ffp_rollout['ID'].map(proba_by_id).fillna(-1)

# --- Sanity checks ---
n_reviewers = ffp_rollout['has_review'].sum()
print(f'Customers with a review: {n_reviewers} / {len(ffp_rollout)}')
print(f'positive_proba range (reviewers): '
      f"[{ffp_rollout.loc[ffp_rollout['has_review'] == 1, 'positive_proba'].min():.4f}, "
      f"{ffp_rollout.loc[ffp_rollout['has_review'] == 1, 'positive_proba'].max():.4f}]")
print(f"Non-reviewer positive_proba unique values: "
      f"{ffp_rollout.loc[ffp_rollout['has_review'] == 0, 'positive_proba'].unique()}")
ffp_rollout[['ID', 'has_review', 'positive_proba']].head()

Customers with a review: 1501 / 20000
positive_proba range (reviewers): [0.0000, 1.0000]
Non-reviewer positive_proba unique values: [-1.]


,ID,has_review,positive_proba
0,20001,0,-1.000000e+00
1,20002,1,5.138800e-08
2,20003,0,-1.000000e+00
3,20004,0,-1.000000e+00
4,20005,0,-1.000000e+00


### 5.1 Do the text features help?

We don't just bolt `has_review` and `positive_proba` onto the already-tuned `xgb_final_model` and refit — its hyperparameters (`max_depth=3`, `scale_pos_weight=1`, etc.) were chosen by the nested search for the old, smaller feature set, and there's no guarantee they're still the right choice for a wider one. To get an honest answer to "do these features help," we re-run the exact same nested-CV procedure from Section 4, just on `X` expanded with the two new columns, and compare its `avg_profit` to `XGBoost (Tuned, Nested CV)` on equal footing.

In [21]:
# --- Nested CV on the expanded feature set (same procedure as Section 4) ---
feature_cols_text = feature_cols + ['has_review', 'positive_proba']
X_text = ffp_train[feature_cols_text]

nested_cv_results_text = cross_validate(
    xgb_search, X_text, y,
    cv=outer_cv,
    scoring=scoring,
    return_estimator=True,
    n_jobs=1,
)

print('Best hyperparameters per outer fold (with text features):')
for fold_idx, fitted_search in enumerate(nested_cv_results_text['estimator']):
    print(f'  Fold {fold_idx}: {fitted_search.best_params_}')

results['XGBoost + Text Features (Tuned, Nested CV)'] = {
    'avg_profit_mean': nested_cv_results_text['test_avg_profit'].mean(),
    'avg_profit_std':  nested_cv_results_text['test_avg_profit'].std(),
    'auc_mean':        nested_cv_results_text['test_auc'].mean(),
    'auc_std':         nested_cv_results_text['test_auc'].std(),
    'accuracy_mean':   nested_cv_results_text['test_accuracy'].mean(),
    'accuracy_std':    nested_cv_results_text['test_accuracy'].std(),
}

results_df = pd.DataFrame(results).T.sort_values('avg_profit_mean', ascending=False)
print('\n=== Model Comparison incl. Text Features (sorted by Avg Profit) ===')
print(results_df.round(4))

Best hyperparameters per outer fold (with text features):
  Fold 0: {'clf__subsample': 0.8, 'clf__scale_pos_weight': 1, 'clf__n_estimators': 100, 'clf__min_child_weight': 1, 'clf__max_depth': 3, 'clf__learning_rate': 0.05, 'clf__gamma': 0, 'clf__colsample_bytree': 0.7}
  Fold 1: {'clf__subsample': 0.8, 'clf__scale_pos_weight': 1, 'clf__n_estimators': 100, 'clf__min_child_weight': 1, 'clf__max_depth': 3, 'clf__learning_rate': 0.05, 'clf__gamma': 0, 'clf__colsample_bytree': 0.7}
  Fold 2: {'clf__subsample': 0.8, 'clf__scale_pos_weight': 1, 'clf__n_estimators': 100, 'clf__min_child_weight': 1, 'clf__max_depth': 3, 'clf__learning_rate': 0.05, 'clf__gamma': 0, 'clf__colsample_bytree': 0.7}
  Fold 3: {'clf__subsample': 0.8, 'clf__scale_pos_weight': 1, 'clf__n_estimators': 100, 'clf__min_child_weight': 1, 'clf__max_depth': 3, 'clf__learning_rate': 0.05, 'clf__gamma': 0, 'clf__colsample_bytree': 0.7}
  Fold 4: {'clf__subsample': 0.8, 'clf__scale_pos_weight': 1, 'clf__n_estimators': 100, 'clf__

### 5.2 Results: do the text features help?

| Model | avg_profit | AUC | accuracy |
|---|---|---|---|
| XGBoost + Text Features (Tuned, Nested CV) | 2.5936 ± 0.2103 | 0.7581 ± 0.0121 | 0.9138 ± 0.0021 |
| XGBoost (Tuned, Nested CV) | 1.6003 ± 0.2652 | 0.6742 ± 0.0217 | 0.9046 ± 0.0004 |

Adding `has_review` and `positive_proba` lifts profit by **+62%** (1.60 → 2.59), with AUC (0.674 → 0.758) and accuracy (0.905 → 0.914) improving too. All three metrics move together, so this isn't a threshold-crossing artifact like we saw with the custom loss - it's a genuine signal.

Every outer fold's inner search converged on the same hyperparameters as the no-text-features model, so this estimate isn't inflated by reusing the same data to both select and score a config — confirmed by the fact that the full-data refit's `best_score_` matches the nested-CV number exactly.

**Conclusion:** the text-mining features are worth keeping. `xgb_final_model_text` - same architecture and threshold, with review-sentiment features added, replaces `xgb_final_model` as the model we deploy.

In [20]:
# --- Final hyperparameter search on full data, with text features included ---
final_search_text = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring=scoring,
    refit='avg_profit',
    cv=outer_cv,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=0,
)
final_search_text.fit(X_text, y)

print('Final selected hyperparameters (XGBoost + Text Features):')
for k, v in final_search_text.best_params_.items():
    print(f'  {k}: {v}')
print(f"Inner CV avg_profit at these params: {final_search_text.best_score_:.4f}")

xgb_final_model_text = final_search_text.best_estimator_

Final selected hyperparameters (XGBoost + Text Features):
  clf__subsample: 0.8
  clf__scale_pos_weight: 1
  clf__n_estimators: 100
  clf__min_child_weight: 1
  clf__max_depth: 3
  clf__learning_rate: 0.05
  clf__gamma: 0
  clf__colsample_bytree: 0.7
Inner CV avg_profit at these params: 2.5936


## 6. Rollout Scoring — Final Deployment Predictions

We score the unlabelled rollout set (`ffp_rollout_X.csv`, IDs 20001–40000) for deployment. The rollout features are built with the exact same pipeline used on the training data:

1. Recency-weighted `FARE_WEIGHTED` / `POINTS_WEIGHTED` (Section 1), using the same `RECENCY_WEIGHTS`.
2. Negative `CUSTOMER_GRADE` imputed with the **training-set** median (`median_grade` from Section 1) — never recomputed on rollout, to avoid leakage.
3. `has_review` / `positive_proba` text-mining features, already merged onto `ffp_rollout` in Section 5.

Per the Section 5 conclusion, `xgb_final_model_text` — same architecture, standard loss, and threshold as `xgb_final_model`, just trained with the text features included — is the model we deploy. We score it with the business threshold `THRESHOLD = 0.1624`.

In [ ]:
# --- Rollout Scoring: Final Deployment Predictions ---
import os

# 1. Recency-weighted FARE/POINTS aggregation (same transform as Section 1, applied to ffp_train)
ffp_rollout['FARE_WEIGHTED'] = sum(
    w * ffp_rollout[c] for w, c in zip(RECENCY_WEIGHTS, fare_cols)
)
ffp_rollout['POINTS_WEIGHTED'] = sum(
    w * ffp_rollout[c] for w, c in zip(RECENCY_WEIGHTS, points_cols)
)

# 2. Impute negative CUSTOMER_GRADE with the training-set median (frozen from Section 1)
ffp_rollout.loc[ffp_rollout['CUSTOMER_GRADE'] < 0, 'CUSTOMER_GRADE'] = median_grade

# 3. has_review / positive_proba were already merged onto ffp_rollout in Section 5

# --- Select features in the exact order xgb_final_model_text was trained on ---
X_rollout = ffp_rollout[feature_cols_text]

# --- Score with the frozen deployment model ---
rollout_proba = xgb_final_model_text.predict_proba(X_rollout)[:, 1]
rollout_pred = (rollout_proba >= THRESHOLD).astype(int)

# --- Build and save the submission file ---
submission = pd.DataFrame({
    'ID': ffp_rollout['ID'],
    'TARGET_CONTACT': rollout_pred
})

submission_path = 'final_predictions_submission.csv'
submission.to_csv(submission_path, index=False)

print(f'Saved {len(submission)} predictions to {os.path.abspath(submission_path)}')
print(f'Predicted contacts (1): {submission["TARGET_CONTACT"].sum()} / {len(submission)} '
      f'({submission["TARGET_CONTACT"].mean():.2%})')
submission.head()